# 114 — Q-guided latent correction in the simulator

This is a **live correction experiment**: at one Euler step, take one bounded Q-ascent step through the SmolVLA flow latent, then finish denoising and execute the first ten actions. The reference is the exact historical ordinary SmolVLA A10 outcome on the same initial identity. P&P is not invoked at deployment. The Q is trained on P&P-root data, so this is explicitly a **transfer pilot**. It is not the previously proposed supervised PCP MLP.

Run one identity first. Inspect the `q_guidance_telemetry` including Q movement and action RMS. A Q rise alone is not a control improvement; compare simulator successes and failure pairs on the full fixed shard after the smoke test. The two shards cover indices 0–9 and are opt-in.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXTRAS = 'sim'
SETUP_ENV = False
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
import torch
from pnp.smolvla_q_online_experiments import run_q10_latent_guidance_transfer_worker

CHECKPOINT = Path('REPLACE_WITH_NOTEBOOK_109_PREFILL_Q_CHECKPOINT.pt')
SHARD_INDEX = 0  # use 1 in a second runtime for the other half
SHARD_COUNT = 2
EPISODE_LIMIT = 1  # smoke; set None for the fixed full shard
EULER_STEP = 4
LATENT_STEP_RMS = 0.05  # in flow-latent units; pilot value, not tuned
RUN_EVAL = False
assert torch.cuda.is_available(), 'Use a GPU runtime.'
assert CHECKPOINT.is_file(), 'Set CHECKPOINT to notebook 109 root-MC checkpoint.'
print({'checkpoint': str(CHECKPOINT), 'shard': f'{SHARD_INDEX}/{SHARD_COUNT}',
       'episode_limit': EPISODE_LIMIT, 'euler_step': EULER_STEP,
       'latent_step_rms': LATENT_STEP_RMS, 'run': RUN_EVAL})


In [ ]:
if RUN_EVAL:
    run_q10_latent_guidance_transfer_worker(
        checkpoint_path=str(CHECKPOINT), shard_index=SHARD_INDEX,
        shard_count=SHARD_COUNT, episode_limit=EPISODE_LIMIT,
        euler_step=EULER_STEP, latent_step_rms=LATENT_STEP_RMS)
else:
    print('No simulator run. Set RUN_EVAL=True after reviewing the pilot parameters.')
